<a href="https://colab.research.google.com/github/habibur8rahaman/Opt_algorithms_in_FSL/blob/main/Copy_of_fsl_optimization_algos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Creating a High-Scale Data-Heterogeneous Network

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

# Load and Prepare Full Dataset ---
(x_train_full, y_train_full), (x_test_full, y_test_orig) = tf.keras.datasets.fashion_mnist.load_data()

num_clients = 20
num_classes = 10
client_data = [None] * num_clients

# Sort the training data by class into a dictionary of lists/arrays
class_indices = {i: np.where(y_train_full == i)[0] for i in range(num_classes)}

# Defining Client Roles ---
generalist_clients = [0, 1, 2, 3]
specialist_clients = list(range(4, 20)) # Clients 4 through 19

# Assigning Generalist Anchor Clients ---
samples_per_class_generalist = 50 # 500 total samples per generalist

for client_id in generalist_clients:
    client_x, client_y = [], []
    for c in range(num_classes):
        # Pop the indices so they aren't reused by other clients
        selected = class_indices[c][:samples_per_class_generalist]
        class_indices[c] = class_indices[c][samples_per_class_generalist:]

        client_x.append(x_train_full[selected])
        client_y.append(y_train_full[selected])

    x_cat = np.concatenate(client_x)
    y_cat = np.concatenate(client_y)

    # Shuffle and normalize
    shuffler = np.random.permutation(len(x_cat))
    x_processed = x_cat[shuffler].astype("float32") / 255.0
    client_data[client_id] = (x_processed, y_cat[shuffler].flatten())
    print(f"Client {client_id+1:2d} (Generalist): {len(y_cat)} samples, Classes present: {np.unique(y_cat)}")

# Assigning Specialist Clients (4 Classes Only, high volume) ---
# We deliberately assign overlapping pairs so the global dataset covers everything,
# but individual clients are highly biased.
specialist_class_pairs = [
    [0, 1], [1, 2], [2, 3], [3, 4],
    [4, 5], [5, 6], [6, 7], [7, 8],
    [8, 9], [9, 0], [0, 5], [1, 6],
    [2, 7], [3, 8], [4, 9], [0, 9]
]

samples_per_class_specialist = 500 # 1000 total samples per specialist

for idx, client_id in enumerate(specialist_clients):
    assigned_classes = specialist_class_pairs[idx]
    client_x, client_y = [], []

    for c in assigned_classes:
        # Pop the next batch of available data for this class
        selected = class_indices[c][:samples_per_class_specialist]
        class_indices[c] = class_indices[c][samples_per_class_specialist:]

        client_x.append(x_train_full[selected])
        client_y.append(y_train_full[selected])

    x_cat = np.concatenate(client_x)
    y_cat = np.concatenate(client_y)

    # Shuffle and normalize
    shuffler = np.random.permutation(len(x_cat))
    x_processed = x_cat[shuffler].astype("float32") / 255.0
    client_data[client_id] = (x_processed, y_cat[shuffler].flatten())
    print(f"Client {client_id+1:2d} (Specialist): {len(y_cat)} samples, Classes present: {np.unique(y_cat)}")

# Process Test Data ---
x_test = x_test_full.astype("float32") / 255.0
y_test = y_test_orig.flatten()

print("\nData assignment complete.")

2026-03-02 19:44:35.075551: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772480675.271450      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772480675.328650      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772480675.778516      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772480675.778558      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772480675.778561      55 computation_placer.cc:177] computation placer alr

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Client  1 (Generalist): 500 samples, Classes present: [0 1 2 3 4 5 6 7 8 9]
Client  2 (Generalist): 500 samples, Classes present: [0 1 2 3 4 5 6 7 8 9]
Client  3 (Generalist): 500 samples, Classes present: [0 1 2 3 4 5 6 7 8 9]
Client  4 (Generalist): 500 samples, Classes present: [0 1 2 3 4 5 6 7 8 9]
Client  5 (Specialist): 1000 samples, Classes present: [0 1]
Client  6 (Specialist): 1000 samples, Classes present: [1 2]
Client  7 (Specialist): 1000 samples, Classes present: [2 3]
Client  8 (Specialist): 1000 samples, Classes present: [3 4]
Client  9 (Specialist): 1000 samples, Classes present: [4 5]
Client 10 (Specialist): 1000 samples, Classes present: [5 6]
Client 11 (Specialist): 1000 samples, Classes present: [6 7]
Client 12 (Specialist): 1000 samples, Classes present: [7 8]
Client 13 (Specialis

In [ ]:
#AlexNet
from tensorflow.keras import models
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

def build_split_model(input_shape=(28, 28, 1), num_classes=10):

    if high_client == 0:
        with tf.device('/cpu:0'):
            print("using SL approach")
            inputs_client = Input(shape=input_shape)
            # Conv1: 32 filters (original: 96), 5x5, stride=1
            x = Conv2D(32, (5, 5), strides=1, activation='relu', padding='same', name='Conv1')(inputs_client)
            # Conv2: 64 filters (original: 256), 3x3
            x = Conv2D(64, (3, 3), activation='relu', padding='same', name='Conv2')(x)
            x = MaxPooling2D(pool_size=(2, 2), name='Pool2')(x)

            client_output = x  # Output shape will be determined dynamically
            client_model = models.Model(inputs=inputs_client, outputs=client_output)
    else:
        # Switchs to GPU when Client is High-CC (High-CC clients perform whole NN on high-end device (GPU))
        with tf.device('/gpu:0'):
            inputs_client = Input(shape=input_shape)
            # Conv1: 32 filters (original: 96), 5x5, stride=1
            x = Conv2D(32, (5, 5), strides=1, activation='relu', padding='same', name='Conv1')(inputs_client)
            # Conv2: 64 filters (original: 256), 3x3
            x = Conv2D(64, (3, 3), activation='relu', padding='same', name='Conv2')(x)
            x = MaxPooling2D(pool_size=(2, 2), name='Pool2')(x)

            client_output = x  # Output shape will be determined dynamically
            client_model = models.Model(inputs=inputs_client, outputs=client_output)


    ## Offloads work to GPU (acts as Edge Server)
    with tf.device('/gpu:0'):
        # Conv3-5: 64 filters (original: 384/256)
        input_gateway = Input(shape=client_model.output_shape[1:])
        x = Conv2D(64, (3, 3), activation='relu', padding='same', name='Conv3')(input_gateway)
        x = Conv2D(64, (3, 3), activation='relu', padding='same', name='Conv4')(x)
        x = Conv2D(64, (3, 3), activation='relu', padding='same', name='Conv5')(x)
        x = MaxPooling2D(pool_size=(2, 2), name='Pool3')(x)
        # Flatten
        x = Flatten(name='Flatten')(x)
        # Dense layers: 512 units (original: 4096)
        x = Dense(512, activation='relu', name='FC1')(x)
        x = Dropout(0.5)(x)
        x = Dense(512, activation='relu', name='FC2')(x)
        x = Dropout(0.5)(x)
        #outputs = Dense(num_classes, activation='softmax', name='Output')(x)
        outputs = Dense(num_classes, activation=None, name='Output')(x) # Remove softmax
        gateway_model = Model(input_gateway, outputs, name='AlexNet_FashionMNIST_Light')
    return client_model, gateway_model

In [ ]:
epochs = 1
batch_size = 64


def evaluate_global_model(cpu_model, gpu_model):
    batch_size_eval = 512
    num_samples = len(x_test)
    all_preds = []
    for i in range(0, num_samples, batch_size_eval):
        batch_x = x_test[i:i+batch_size_eval]
        x_intermediate = cpu_model(batch_x, training=False)
        logits = gpu_model(x_intermediate, training=False)
        batch_preds = tf.argmax(logits, axis=1)
        all_preds.append(batch_preds)

    final_preds = tf.concat(all_preds, axis=0)
    y_test_tensor = tf.cast(y_test, final_preds.dtype)
    acc = tf.reduce_mean(tf.cast(tf.equal(final_preds, y_test_tensor), tf.float32))
    print(f"Global Test Accuracy: {acc.numpy():.4f}")

In [ ]:
#FedProx

import tensorflow as tf
import numpy as np
import time
import random

global high_client
high_client = 0
high_end_client = []

# --- FedProx Loss ---
def fedprox_loss(logits, y_true, model_vars, global_vars, mu):
    y_true = tf.cast(y_true, tf.int32)
    base_loss = tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y_true, logits=logits)
    )
    proximal_term = 0.0
    for local_w, global_w in zip(model_vars, global_vars):
        proximal_term += tf.nn.l2_loss(local_w - global_w)
    return base_loss + (mu * proximal_term)

# --- Delta Helpers ---
def get_model_deltas(global_weights, local_weights):
    return [local - global_w for local, global_w in zip(local_weights, global_weights)]

def aggregate_deltas_weighted(all_deltas, sample_counts):
    total_samples = sum(sample_counts)
    averaged_deltas = []
    for layer_idx in range(len(all_deltas[0])):
        layer_delta = tf.zeros_like(all_deltas[0][layer_idx])
        for client_idx, client_delta in enumerate(all_deltas):
            weight = sample_counts[client_idx] / total_samples
            layer_delta += client_delta[layer_idx] * weight
        averaged_deltas.append(layer_delta)
    return averaged_deltas


# System Initialization ---
num_rounds = epochs
MU = 0.1              # FedProx penalty constant
fraction_fit = 0.4    # ADJUSTMENT 3: 40% of clients per round

num_clients_per_round = max(1, int(num_clients * fraction_fit))

global_cpu_model, global_gpu_model = build_split_model()

cpu_models = []
gpu_models = []
optimizers = []

for i in range(num_clients):
    c_cpu, c_gpu = build_split_model()
    cpu_models.append(c_cpu)
    gpu_models.append(c_gpu)
    optimizers.append(tf.keras.optimizers.Adam(learning_rate=0.001))

# --- Main FedProx Training Loop ---
for round_idx in range(num_rounds):
    start_time = time.time()
    print(f"\n--- FedProx Round {round_idx + 1} ---")

    global_cpu_weights = global_cpu_model.get_weights()
    global_gpu_weights = global_gpu_model.get_weights()

    round_deltas_cpu = []
    round_deltas_gpu = []
    sample_counts = []

    # ADJUSTMENT 3: Selecting a random subset of clients for this round
    selected_clients = random.sample(range(num_clients), num_clients_per_round)

    for client_id in selected_clients:
        client_start_time = time.time()

        cpu_model = cpu_models[client_id]
        gpu_model = gpu_models[client_id]

        cpu_model.set_weights(global_cpu_weights)
        gpu_model.set_weights(global_gpu_weights)

        x_client, y_client = client_data[client_id]
        sample_counts.append(len(x_client))
        train_ds = tf.data.Dataset.from_tensor_slices((x_client, y_client)).batch(batch_size)

        # ADJUSTMENT 1: Multiple local epochs to cause client drift
        client_local_epochs = random.randint(1, 5)

        for _ in range(client_local_epochs):
            # ... local training ...
            for x_batch, y_batch in train_ds:
                with tf.GradientTape() as tape:
                    x_inter = cpu_model(x_batch, training=True)
                    logits = gpu_model(x_inter, training=True)

                    local_vars = cpu_model.trainable_variables + gpu_model.trainable_variables
                    global_vars = global_cpu_weights + global_gpu_weights

                    # ADJUSTMENT 2: FedProx Loss applied
                    loss = fedprox_loss(logits, y_batch, local_vars, global_vars, MU)

                grads = tape.gradient(loss, local_vars)
                optimizers[client_id].apply_gradients(zip(grads, local_vars))

        delta_cpu = get_model_deltas(global_cpu_weights, cpu_model.get_weights())
        delta_gpu = get_model_deltas(global_gpu_weights, gpu_model.get_weights())

        round_deltas_cpu.append(delta_cpu)
        round_deltas_gpu.append(delta_gpu)

        print(f"Client {client_id} updated. Time: {time.time() - client_start_time:.2f}s")

    avg_delta_cpu = aggregate_deltas_weighted(round_deltas_cpu, sample_counts)
    avg_delta_gpu = aggregate_deltas_weighted(round_deltas_gpu, sample_counts)

    new_global_cpu = [g + d for g, d in zip(global_cpu_weights, avg_delta_cpu)]
    new_global_gpu = [g + d for g, d in zip(global_gpu_weights, avg_delta_gpu)]

    global_cpu_model.set_weights(new_global_cpu)
    global_gpu_model.set_weights(new_global_gpu)

    evaluate_global_model(global_cpu_model, global_gpu_model)
    print(f"Round {round_idx + 1} complete. Total Time: {time.time() - start_time:.2f}s")

In [ ]:
#FedDyn

import tensorflow as tf
import numpy as np
import time
import random

global high_client
high_client = 0
high_end_client = []

# --- 1. FedDyn Loss Function ---
def feddyn_loss(logits, y_true, model_vars, global_vars, client_h_vars, alpha):
    """
    FedDyn Loss: CrossEntropy - <h_i, w> + (alpha/2) * ||w - w_t||^2
    """
    y_true = tf.cast(y_true, tf.int32)
    base_loss = tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y_true, logits=logits)
    )

    # Linear Penalty: Dot product of the client's historical gradient and current weights
    linear_penalty = 0.0
    for w, h in zip(model_vars, client_h_vars):
        linear_penalty += tf.reduce_sum(w * h)

    # Proximal Term: Keeps the model from drifting too fast (similar to FedProx)
    proximal_term = 0.0
    for local_w, global_w in zip(model_vars, global_vars):
        proximal_term += tf.nn.l2_loss(local_w - global_w)

    return base_loss - linear_penalty + (alpha * proximal_term)

# --- 2. Delta Helpers ---
def get_model_deltas(global_weights, local_weights):
    return [local - global_w for local, global_w in zip(local_weights, global_weights)]

def aggregate_deltas_simple(all_deltas):
    """FedDyn typically relies on simple averaging for mathematical guarantees."""
    num_participating = len(all_deltas)
    averaged_deltas = []
    for layer_idx in range(len(all_deltas[0])):
        layer_sum = tf.reduce_sum([client_delta[layer_idx] for client_delta in all_deltas], axis=0)
        averaged_deltas.append(layer_sum / float(num_participating))
    return averaged_deltas


# System Initialization ---
num_rounds = epochs
fraction_fit = 0.2
num_clients_per_round = max(1, int(num_clients * fraction_fit))
ALPHA = 0.01  # FedDyn regularization parameter

global_cpu_model, global_gpu_model = build_split_model()

cpu_models, gpu_models, optimizers = [], [], []

# ==========================================
# NEW: FedDyn Historical Gradient Trackers
# ==========================================
# 1. Global Server State
global_h_cpu = [tf.zeros_like(w) for w in global_cpu_model.get_weights()]
global_h_gpu = [tf.zeros_like(w) for w in global_gpu_model.get_weights()]

# 2. Local Client States for all 20 clients
client_h_cpu = [[tf.zeros_like(w) for w in global_cpu_model.get_weights()] for _ in range(num_clients)]
client_h_gpu = [[tf.zeros_like(w) for w in global_gpu_model.get_weights()] for _ in range(num_clients)]

for i in range(num_clients):
    c_cpu, c_gpu = build_split_model()
    cpu_models.append(c_cpu)
    gpu_models.append(c_gpu)
    optimizers.append(tf.keras.optimizers.Adam(learning_rate=0.001))


# Main FedDyn Training Loop ---
for round_idx in range(num_rounds):
    start_time = time.time()
    print(f"\n--- FedDyn Round {round_idx + 1} ---")

    global_cpu_weights = global_cpu_model.get_weights()
    global_gpu_weights = global_gpu_model.get_weights()

    round_deltas_cpu = []
    round_deltas_gpu = []

    selected_clients = random.sample(range(num_clients), num_clients_per_round)

    for client_id in selected_clients:
        client_start_time = time.time()

        cpu_model = cpu_models[client_id]
        gpu_model = gpu_models[client_id]

        cpu_model.set_weights(global_cpu_weights)
        gpu_model.set_weights(global_gpu_weights)

        x_client, y_client = client_data[client_id]
        train_ds = tf.data.Dataset.from_tensor_slices((x_client, y_client)).batch(batch_size)

        # Retrieve this specific client's historical gradient
        h_i_cpu = client_h_cpu[client_id]
        h_i_gpu = client_h_gpu[client_id]


        client_local_epochs = random.randint(1, 5)e

        for _ in range(client_local_epochs):
            for x_batch, y_batch in train_ds:
                with tf.GradientTape() as tape:
                    x_inter = cpu_model(x_batch, training=True)
                    logits = gpu_model(x_inter, training=True)

                    local_vars = cpu_model.trainable_variables + gpu_model.trainable_variables
                    global_vars = global_cpu_weights + global_gpu_weights
                    client_h_vars = h_i_cpu + h_i_gpu

                    # Apply FedDyn Loss
                    loss = feddyn_loss(logits, y_batch, local_vars, global_vars, client_h_vars, ALPHA)

                grads = tape.gradient(loss, local_vars)
                optimizers[client_id].apply_gradients(zip(grads, local_vars))

        # 1. Compute Local Deltas
        delta_cpu = get_model_deltas(global_cpu_weights, cpu_model.get_weights())
        delta_gpu = get_model_deltas(global_gpu_weights, gpu_model.get_weights())

        round_deltas_cpu.append(delta_cpu)
        round_deltas_gpu.append(delta_gpu)

        # Update Client's Local Historical Gradient (h_i = h_i - alpha * delta)
        client_h_cpu[client_id] = [h - ALPHA * d for h, d in zip(h_i_cpu, delta_cpu)]
        client_h_gpu[client_id] = [h - ALPHA * d for h, d in zip(h_i_gpu, delta_gpu)]

        print(f"  Client {client_id} updated. Time: {time.time() - client_start_time:.2f}s")

    # --- Server Aggregation & FedDyn Math ---

    # 1. Average the deltas of participating clients
    avg_delta_cpu = aggregate_deltas_simple(round_deltas_cpu)
    avg_delta_gpu = aggregate_deltas_simple(round_deltas_gpu)

    # 2. Sum the deltas to update the Global Historical Gradient (h)
    sum_delta_cpu = [d * len(selected_clients) for d in avg_delta_cpu]
    sum_delta_gpu = [d * len(selected_clients) for d in avg_delta_gpu]

    # Update Global h: h = h - (alpha / Total_Clients) * sum(deltas)
    global_h_cpu = [h - (ALPHA / num_clients) * s for h, s in zip(global_h_cpu, sum_delta_cpu)]
    global_h_gpu = [h - (ALPHA / num_clients) * s for h, s in zip(global_h_gpu, sum_delta_gpu)]

    # 3. Final Global Weight Update: w = w + avg_delta - (1/alpha) * global_h
    new_global_cpu = [w + d - (1.0 / ALPHA) * h for w, d, h in zip(global_cpu_weights, avg_delta_cpu, global_h_cpu)]
    new_global_gpu = [w + d - (1.0 / ALPHA) * h for w, d, h in zip(global_gpu_weights, avg_delta_gpu, global_h_gpu)]

    global_cpu_model.set_weights(new_global_cpu)
    global_gpu_model.set_weights(new_global_gpu)

    evaluate_global_model(global_cpu_model, global_gpu_model)
    print(f"Round {round_idx + 1} complete. Total Time: {time.time() - start_time:.2f}s")

In [ ]:
#FedOpt/ FedAdam

import tensorflow as tf
import numpy as np
import time
import random


global high_client
high_client = 0
high_end_client = []
# --- Pure FedAvg Loss (FedAdam handles optimization on the server) ---
def fedavg_loss(logits, y_true):
    y_true = tf.cast(y_true, tf.int32)
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y_true, logits=logits)
    )

# --- Delta Helpers ---
def get_model_deltas(global_weights, local_weights):
    return [local - global_w for local, global_w in zip(local_weights, global_weights)]

def aggregate_deltas_weighted(all_deltas, sample_counts):
    total_samples = sum(sample_counts)
    averaged_deltas = []
    for layer_idx in range(len(all_deltas[0])):
        layer_delta = tf.zeros_like(all_deltas[0][layer_idx])
        for client_idx, client_delta in enumerate(all_deltas):
            weight = sample_counts[client_idx] / total_samples
            layer_delta += client_delta[layer_idx] * weight
        averaged_deltas.append(layer_delta)
    return averaged_deltas



# --- System Initialization ---
num_rounds = epochs
fraction_fit = 0.2
num_clients_per_round = max(1, int(num_clients * fraction_fit))

global_cpu_model, global_gpu_model = build_split_model()

cpu_models = []
gpu_models = []
optimizers = []

for i in range(num_clients):
    c_cpu, c_gpu = build_split_model()
    cpu_models.append(c_cpu)
    gpu_models.append(c_gpu)
    optimizers.append(tf.keras.optimizers.Adam(learning_rate=0.001))

# ==========================================
# NEW: FedAdam Server-Side Variables
# ==========================================
BETA_1 = 0.9
BETA_2 = 0.999
SERVER_LR = 0.01  # Server learning rate (eta)
TAU = 1e-3        # Epsilon for numerical stability

# Initialize momentum and variance trackers with zeros
m_cpu = [tf.zeros_like(w) for w in global_cpu_model.get_weights()]
v_cpu = [tf.zeros_like(w) for w in global_cpu_model.get_weights()]
m_gpu = [tf.zeros_like(w) for w in global_gpu_model.get_weights()]
v_gpu = [tf.zeros_like(w) for w in global_gpu_model.get_weights()]

# --- Main FedAdam Training Loop ---
for round_idx in range(num_rounds):
    start_time = time.time()
    print(f"\n--- FedAdam Round {round_idx + 1} ---")

    global_cpu_weights = global_cpu_model.get_weights()
    global_gpu_weights = global_gpu_model.get_weights()

    round_deltas_cpu = []
    round_deltas_gpu = []
    sample_counts = []

    selected_clients = random.sample(range(num_clients), num_clients_per_round)

    for client_id in selected_clients:
        client_start_time = time.time()

        cpu_model = cpu_models[client_id]
        gpu_model = gpu_models[client_id]

        cpu_model.set_weights(global_cpu_weights)
        gpu_model.set_weights(global_gpu_weights)

        x_client, y_client = client_data[client_id]
        sample_counts.append(len(x_client))
        train_ds = tf.data.Dataset.from_tensor_slices((x_client, y_client)).batch(batch_size)


        client_local_epochs = random.randint(1, 5)

        for _ in range(client_local_epochs):
            for x_batch, y_batch in train_ds:
                with tf.GradientTape() as tape:
                    x_inter = cpu_model(x_batch, training=True)
                    logits = gpu_model(x_inter, training=True)
                    loss = fedavg_loss(logits, y_batch)

                local_vars = cpu_model.trainable_variables + gpu_model.trainable_variables
                grads = tape.gradient(loss, local_vars)
                optimizers[client_id].apply_gradients(zip(grads, local_vars))

        delta_cpu = get_model_deltas(global_cpu_weights, cpu_model.get_weights())
        delta_gpu = get_model_deltas(global_gpu_weights, gpu_model.get_weights())

        round_deltas_cpu.append(delta_cpu)
        round_deltas_gpu.append(delta_gpu)

        print(f"  Client {client_id} updated. Time: {time.time() - client_start_time:.2f}s")

    avg_delta_cpu = aggregate_deltas_weighted(round_deltas_cpu, sample_counts)
    avg_delta_gpu = aggregate_deltas_weighted(round_deltas_gpu, sample_counts)

    # ==========================================
    # NEW: Server-Side Adam Optimization (FedAdam)
    # Treats negative delta as the pseudo-gradient
    # ==========================================
    def apply_fedadam(global_w, avg_delta, m, v):
        new_w, new_m, new_v = [], [], []
        for g_w, d, m_t, v_t in zip(global_w, avg_delta, m, v):
            pseudo_grad = -d

            # Momentum
            m_next = BETA_1 * m_t + (1 - BETA_1) * pseudo_grad
            # Variance
            v_next = BETA_2 * v_t + (1 - BETA_2) * tf.square(pseudo_grad)
            # Update
            w_next = g_w - SERVER_LR * m_next / (tf.sqrt(v_next) + TAU)

            new_w.append(w_next)
            new_m.append(m_next)
            new_v.append(v_next)
        return new_w, new_m, new_v

    # Applying the math to both CPU (client) and GPU (edge server) model segments
    new_global_cpu, m_cpu, v_cpu = apply_fedadam(global_cpu_weights, avg_delta_cpu, m_cpu, v_cpu)
    new_global_gpu, m_gpu, v_gpu = apply_fedadam(global_gpu_weights, avg_delta_gpu, m_gpu, v_gpu)

    global_cpu_model.set_weights(new_global_cpu)
    global_gpu_model.set_weights(new_global_gpu)

    evaluate_global_model(global_cpu_model, global_gpu_model)
    print(f"Round {round_idx + 1} complete. Total Time: {time.time() - start_time:.2f}s")

In [ ]:
#FedAvg

import tensorflow as tf
import numpy as np
import time
import random

# --- Pure FedAvg Loss ---
def fedavg_loss(logits, y_true):
    y_true = tf.cast(y_true, tf.int32)
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y_true, logits=logits)
    )

# --- Delta Helpers ---
def get_model_deltas(global_weights, local_weights):
    return [local - global_w for local, global_w in zip(local_weights, global_weights)]

def aggregate_deltas_weighted(all_deltas, sample_counts):
    total_samples = sum(sample_counts)
    averaged_deltas = []
    for layer_idx in range(len(all_deltas[0])):
        layer_delta = tf.zeros_like(all_deltas[0][layer_idx])
        for client_idx, client_delta in enumerate(all_deltas):
            weight = sample_counts[client_idx] / total_samples
            layer_delta += client_delta[layer_idx] * weight
        averaged_deltas.append(layer_delta)
    return averaged_deltas



# ystem Initialization ---
num_rounds = epochs
fraction_fit = 0.4     # 40% of clients per round


num_clients_per_round = max(1, int(num_clients * fraction_fit))

global_cpu_model, global_gpu_model = build_split_model()

cpu_models = []
gpu_models = []
optimizers = []

for i in range(num_clients):
    c_cpu, c_gpu = build_split_model()
    cpu_models.append(c_cpu)
    gpu_models.append(c_gpu)
    optimizers.append(tf.keras.optimizers.Adam(learning_rate=0.001))

# Main FedAvg Training Loop ---
for round_idx in range(num_rounds):
    start_time = time.time()
    print(f"\n--- FedAvg Round {round_idx + 1} ---")

    global_cpu_weights = global_cpu_model.get_weights()
    global_gpu_weights = global_gpu_model.get_weights()

    round_deltas_cpu = []
    round_deltas_gpu = []
    sample_counts = []

    # Selecting a random subset of clients for this round
    selected_clients = random.sample(range(num_clients), num_clients_per_round)

    for client_id in selected_clients:
        client_start_time = time.time()

        cpu_model = cpu_models[client_id]
        gpu_model = gpu_models[client_id]

        cpu_model.set_weights(global_cpu_weights)
        gpu_model.set_weights(global_gpu_weights)

        x_client, y_client = client_data[client_id]
        sample_counts.append(len(x_client))
        train_ds = tf.data.Dataset.from_tensor_slices((x_client, y_client)).batch(batch_size)


        client_local_epochs = random.randint(1, 5)

        for _ in range(client_local_epochs):

            for x_batch, y_batch in train_ds:
                with tf.GradientTape() as tape:
                    x_inter = cpu_model(x_batch, training=True)
                    logits = gpu_model(x_inter, training=True)

                    loss = fedavg_loss(logits, y_batch)

                local_vars = cpu_model.trainable_variables + gpu_model.trainable_variables
                grads = tape.gradient(loss, local_vars)
                optimizers[client_id].apply_gradients(zip(grads, local_vars))

        delta_cpu = get_model_deltas(global_cpu_weights, cpu_model.get_weights())
        delta_gpu = get_model_deltas(global_gpu_weights, gpu_model.get_weights())

        round_deltas_cpu.append(delta_cpu)
        round_deltas_gpu.append(delta_gpu)

        print(f"Client {client_id} updated. Time: {time.time() - client_start_time:.2f}s")

    avg_delta_cpu = aggregate_deltas_weighted(round_deltas_cpu, sample_counts)
    avg_delta_gpu = aggregate_deltas_weighted(round_deltas_gpu, sample_counts)

    new_global_cpu = [g + d for g, d in zip(global_cpu_weights, avg_delta_cpu)]
    new_global_gpu = [g + d for g, d in zip(global_gpu_weights, avg_delta_gpu)]

    global_cpu_model.set_weights(new_global_cpu)
    global_gpu_model.set_weights(new_global_gpu)

    evaluate_global_model(global_cpu_model, global_gpu_model)
    print(f"Round {round_idx + 1} complete. Total Time: {time.time() - start_time:.2f}s")